### 1. Start a SparkSession

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Bronze") \
    .master("local[*]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/16 10:49:11 WARN Utils: Your hostname, MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.12 instead (on interface en0)
26/02/16 10:49:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/16 10:49:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### 2. Define schemas

In [2]:
from pyspark.sql.types import *

ratings_schema = StructType([
    StructField("userId",       IntegerType(),  True),
    StructField("movieId",      IntegerType(),  True),
    StructField("rating",       FloatType(),    True),
    StructField("timestamp",    LongType(),     True),
])

movies_schema = StructType([
    StructField("movieId",  IntegerType(),  True),
    StructField("title",    StringType(),   True),
    StructField("genres",   StringType(),   True),
])

links_schema = StructType([
    StructField("movieId",  IntegerType(),  True),
    StructField("imdbId",   IntegerType(),  True),
    StructField("tmdbId",   IntegerType(),  True),
])

tags_schema = StructType([
    StructField("userId",       IntegerType(),  True),
    StructField("movieId",      IntegerType(),  True),
    StructField("tag",          StringType(),   True),
    StructField("timestamp",    LongType(),     True),
])

### 3. Read each CSV

In [3]:
df_ratings = spark.read.csv("ml-32m/ratings.csv", header=True, schema=ratings_schema)
df_movies  = spark.read.csv("ml-32m/movies.csv",  header=True, schema=movies_schema)
df_links   = spark.read.csv("ml-32m/links.csv",   header=True, schema=links_schema)
df_tags    = spark.read.csv("ml-32m/tags.csv",     header=True, schema=tags_schema)

### 4. Add ingestion metadata

In [4]:
from pyspark.sql.functions import current_timestamp, lit

def add_metadata(df, source_name):
    return df \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_source_file", lit(source_name))

### 5. Write to /bronze as parquet

In [5]:
add_metadata(df_ratings, "ratings.csv") \
    .write.mode("overwrite").parquet("bronze/ratings")

add_metadata(df_movies, "movies.csv") \
    .write.mode("overwrite").parquet("bronze/movies")

add_metadata(df_links, "links.csv") \
    .write.mode("overwrite").parquet("bronze/links")

add_metadata(df_tags, "tags.csv") \
    .write.mode("overwrite").parquet("bronze/tags")

26/02/16 10:49:19 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers
26/02/16 10:49:19 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 84,44% for 9 writers
26/02/16 10:49:19 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 76,00% for 10 writers
26/02/16 10:49:19 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 69,09% for 11 writers
26/02/16 10:49:19 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 63,33% for 12 writers
26/02/16 10:49:22 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 69,09% for 11 writers
26/02/16 10:49:22 WARN MemoryManager: Total allocation exceeds 95,

### 6. Print sanity checks

In [6]:
for name, df in [("ratings", df_ratings), ("movies", df_movies),
                  ("links", df_links), ("tags", df_tags)]:
    print(f"\n=== {name} ===")
    print(f"Rows: {df.count()}")
    df.printSchema()
    df.show(3)


=== ratings ===
Rows: 32000204
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: float (nullable = true)
 |-- timestamp: long (nullable = true)

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|     17|   4.0|944249077|
|     1|     25|   1.0|944250228|
|     1|     29|   2.0|943230976|
+------+-------+------+---------+
only showing top 3 rows

=== movies ===
Rows: 87585
root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
+-------+--------------------+--------------------+
only showing top 3 rows

=== lin